# AROME Daily Fetch — France Metropolitan Grid

Downloads the AROME 00Z run for **all of metropolitan France** and stores
processed daily gridded NetCDF files. Any parcel in France can then extract
from this cache via `extract_parcel_forcing()` — no per-parcel download needed.

**Run once per day** (morning, AROME 00Z available ~04:00 UTC).

Output structure:
```
data/arome_daily/
    arome_daily_2026-07-15.nc    ← analysis quality (steps 0-23h)
    arome_daily_2026-07-16.nc    ← forecast quality (steps 24-47h)
    ...
```

File sizes: ~50-100 MB/day for the daily aggregated grid over France.
Raw GRIB (~500 MB+) is deleted after processing.

In [ ]:
import logging
import os
from datetime import date, timedelta
from pathlib import Path
from datetime import datetime, timedelta, timezone

import numpy as np
import pandas as pd
import xarray as xr

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s")
logger = logging.getLogger("arome_daily")

In [2]:
# === Configuration ===

FETCH_DATE = date.today()
# FETCH_DATE = date(2026, 7, 15)  # override for testing

# France metropolitan bounding box (generous margins for interpolation)
FRANCE_BBOX = {
    "north": 51.5,   # north of Dunkerque
    "south": 41.0,   # south of Corsica
    "west": -6.0,    # west of Brittany
    "east": 10.0,    # east of Corsica
}

# Storage
AROME_DAILY_DIR = Path("data/arome_daily")
AROME_RAW_DIR = Path("data/arome_raw")  # temporary, deleted after processing
AROME_DAILY_DIR.mkdir(parents=True, exist_ok=True)
AROME_RAW_DIR.mkdir(parents=True, exist_ok=True)

# Delete raw GRIB after processing to save disk
DELETE_RAW = True

print(f"Fetch date: {FETCH_DATE}")
print(f"Output dir: {AROME_DAILY_DIR}")

Fetch date: 2026-06-12
Output dir: data/arome_daily


# Step 0 : Get Capabilities

In [5]:
from irrigator.utils.auth_meteofrance import meteo_headers
import requests


def get_arome_capabilities() -> str:
    base_url = "https://public-api.meteofrance.fr/public/arome/1.0"
    resource = "wcs/MF-NWP-HIGHRES-AROME-001-FRANCE-WCS"

    resp = requests.get(
        f"{base_url}/{resource}/GetCapabilities",
        params={
            "service": "WCS",
            "version": "2.0.1",
            "language": "fre",
        },
        headers=meteo_headers(),
        timeout=60,
    )
    resp.raise_for_status()
    return resp.text


xml = get_arome_capabilities()
AVAILABLE_COVERAGES = []
# crude check
for line in xml.splitlines():
    if "CoverageId" in line:
        AVAILABLE_COVERAGES.append(
            line.strip().replace("<wcs:CoverageId>", "").replace("</wcs:CoverageId>", "")
        )
print(AVAILABLE_COVERAGES[:10])

['GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T00.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T03.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T06.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T09.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T12.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T15.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T18.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-08T21.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-09T00.00.00Z', 'GEOMETRIC_HEIGHT__GROUND_OR_WATER_SURFACE___2026-06-09T03.00.00Z']


---
## Step 1: Download raw AROME 00Z

There are many available datasets, we rely on: \
1 - For hourly accumulated precipitation : "TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE___2026-06-10T00.00.00Z_PT1H" \
2 - For Wind at 10m :
coverage_id = "WIND_SPEED__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND___2026-06-06T00.00.00Z" 
and pass height=10 \
3 - For Dewpoint temperature at 2m :
DEW_POINT_TEMPERATURE__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND___2026-06-06T00.00.00Z
and pass height=2 \
4 - For Temperature at 2m :
TEMPERATURE__GROUND_OR_WATER_SURFACE___2026-06-06T00.00.00Z
and pass height=2 \
5 - For Solar Radiation :
DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE___2026-06-06T15.00.00Z_PT6H

In [ ]:
VARIABLES = {
    "hourly_precip": "TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE",
    "wind_10m": "WIND_SPEED__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND",
    "dewpoint_2m": "DEW_POINT_TEMPERATURE__SPECIFIC_HEIGHT_LEVEL_ABOVE_GROUND",
    "temp_2m": "TEMPERATURE__GROUND_OR_WATER_SURFACE",
    "solar_rad": "DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE",
}
def format_coverage(variable : str, date : str, accum_period : str | None = None):
    "Date should be like '2026-09-21T09'"
    "Accum_period like PT1H, P1D, ... for accumulation variables"
    return (
        variable
        + "___"
        + date 
        + ".00.00Z_"
        + accum_period
    )

In [73]:
from datetime import date
from pathlib import Path
import requests
import re
from datetime import datetime, timedelta, timezone

from irrigator.utils.auth_meteofrance import meteo_headers


def fetch_arome_france(
    target_date: date,
    coverage_id : str,
    run_ahead: int = 0,
    bbox: dict = FRANCE_BBOX,
    *,
    overwrite : bool = False,
    height : str | None = None,
) -> Path:
    """Download AROME snow depth over metropolitan France as GRIB2."""


    out_path = AROME_RAW_DIR / f"arome_france/{coverage_id}_ahead_{run_ahead}H.grib2"

    if out_path.exists() and not overwrite:
        logger.info("Already downloaded: %s", out_path)
        return out_path

    session = requests.Session()
    session.headers.update(meteo_headers())

    base_url = "https://public-api.meteofrance.fr/public/arome/1.0"
    resource = "wcs/MF-NWP-HIGHRES-AROME-001-FRANCE-WCS"

    date_coverage = re.sub(".*__", "", coverage_id)
    date_coverage = re.sub("_.*", "", date_coverage)
    dt = datetime.strptime(date_coverage, "%Y-%m-%dT%H.%M.%SZ").replace(tzinfo=timezone.utc)
    run_hour = (dt + timedelta(hours=run_ahead)).strftime("%Y-%m-%dT%H:%M:%SZ")

    params = {
        "service": "WCS",
        "version": "2.0.1",
        "coverageId": coverage_id,
        "subset": [
            f"time({run_hour})",
            f"lat({bbox['south']},{bbox['north']})",
            f"long({bbox['west']},{bbox['east']})",
        ],
        "format": "application/wmo-grib",
    }

    if "HEIGHT_LEVEL" in coverage_id:
        params["subset"].append(f"height({height})") 

    logger.info(f"Downloading AROME France from {target_date} for {run_ahead} ahead")

    resp = session.get(
        f"{base_url}/{resource}/GetCoverage",
        params=params,
        timeout=600,
    )

    if not resp.ok:
        print("URL:", resp.url)
        print("Status:", resp.status_code)
        print("Body:", resp.text[:4000])
        resp.raise_for_status()

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_bytes(resp.content)

    logger.info("Downloaded: %s (%.0f MB)", out_path, len(resp.content) / 1e6)
    return out_path


coverage_id = "TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE___2026-06-10T00.00.00Z_P1D"
#coverage_id = "TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE___2026-06-10T00.00.00Z_PT1H"
coverage_id = (
    "DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE___2026-06-06T15.00.00Z_P1D"
)
raw_path = fetch_arome_france(FETCH_DATE,coverage_id=coverage_id, run_ahead=30, bbox=FRANCE_BBOX, overwrite=True, height = "2")
print(f"Raw GRIB: {raw_path} ({raw_path.stat().st_size / 1e6:.0f} MB)")

2026-06-10 16:29:32,344 Downloading AROME France from 2026-06-10 for 30 ahead
2026-06-10 16:29:35,289 Downloaded: data/arome_raw/arome_france/DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE___2026-06-06T15.00.00Z_P1D_ahead_30H.grib2 (3 MB)


Raw GRIB: data/arome_raw/arome_france/DOWNWARD_SHORT_WAVE_RADIATION_FLUX__GROUND_OR_WATER_SURFACE___2026-06-06T15.00.00Z_P1D_ahead_30H.grib2 (3 MB)


---
## Step 2: Open and standardize

In [ ]:
from irrigator.ingestion.meteofrance_client import open_forecast
from irrigator.forecasts.short_term import standardize_forecast_to_era5_format

ds_raw = open_forecast(raw_path)
ds = standardize_forecast_to_era5_format(ds_raw, source="arome")

# Ensure valid_time is computed from step
if "valid_time" not in ds.dims and "step" in ds.dims:
    ref = ds.time if "time" in ds.coords else np.datetime64(f"{FETCH_DATE}T00:00:00")
    ds = ds.assign_coords(valid_time=ref + ds.step).swap_dims({"step": "valid_time"})

print(f"Grid: {ds.sizes}")
print(f"Variables: {list(ds.data_vars)}")
print(f"Time range: {pd.Timestamp(ds.valid_time.values[0])} → {pd.Timestamp(ds.valid_time.values[-1])}")

---
## Step 3: De-accumulate precipitation and radiation

In [ ]:
for acc_var in ["precip_mm", "rs_mj"]:
    if acc_var in ds:
        raw_total = float(ds[acc_var].mean())
        diffed = ds[acc_var].diff(dim="valid_time").clip(min=0)
        first = ds[acc_var].isel(valid_time=0).clip(min=0)
        ds[acc_var] = xr.concat([first, diffed], dim="valid_time")
        new_total = float(ds[acc_var].mean())
        print(f"{acc_var}: de-accumulated (spatial mean: {raw_total:.4f} raw → {new_total:.4f} per-step)")

---
## Step 4: Aggregate to daily and save — one file per calendar day

In [ ]:
def aggregate_and_save_day(ds_hourly, cal_date, out_dir, source_label):
    """Aggregate sub-daily grid to daily, save as NetCDF.
    
    Keeps the full spatial grid (all of France).
    Variables match ERA5-Land daily format for extract_parcel_forcing().
    """
    out_path = out_dir / f"arome_daily_{cal_date.isoformat()}.nc"
    
    if out_path.exists():
        logger.info("Already processed: %s", out_path.name)
        return out_path
    
    # Filter to this calendar day
    day_mask = pd.DatetimeIndex(ds_hourly.valid_time.values).date == cal_date
    ds_day = ds_hourly.isel(valid_time=day_mask)
    
    if ds_day.sizes["valid_time"] == 0:
        logger.warning("No data for %s", cal_date)
        return None
    
    daily_vars = {}
    
    # Temperature
    if "t_mean" in ds_day:
        daily_vars["t_mean"] = ds_day["t_mean"].mean(dim="valid_time")
        daily_vars["t_min"] = ds_day["t_mean"].min(dim="valid_time")
        daily_vars["t_max"] = ds_day["t_mean"].max(dim="valid_time")
    if "t_min" in ds_day:
        daily_vars["t_min"] = ds_day["t_min"].min(dim="valid_time")
    if "t_max" in ds_day:
        daily_vars["t_max"] = ds_day["t_max"].max(dim="valid_time")
    
    # Dewpoint, wind, pressure: daily mean
    for var in ["dewpoint", "wind_speed_10m", "pressure_kpa"]:
        if var in ds_day:
            daily_vars[var] = ds_day[var].mean(dim="valid_time")
    
    # Precipitation, radiation: daily sum (already de-accumulated)
    for var in ["precip_mm", "rs_mj"]:
        if var in ds_day:
            daily_vars[var] = ds_day[var].sum(dim="valid_time").clip(min=0)
    
    # Build single-timestep dataset
    daily_ds = xr.Dataset(
        {k: v.expand_dims(valid_time=[np.datetime64(cal_date)]) for k, v in daily_vars.items()}
    )
    daily_ds.attrs["source"] = f"AROME 00Z {FETCH_DATE} ({source_label})"
    daily_ds.attrs["spatial_coverage"] = "France metropolitan"
    
    # Compress for smaller files
    encoding = {v: {"zlib": True, "complevel": 4} for v in daily_ds.data_vars}
    daily_ds.to_netcdf(out_path, encoding=encoding)
    
    size_mb = out_path.stat().st_size / 1e6
    logger.info("Saved %s: %s (%.1f MB)", source_label, out_path.name, size_mb)
    return out_path

In [ ]:
# Split into calendar days and process each
analysis_date = FETCH_DATE
forecast_date = FETCH_DATE + timedelta(days=1)

# Day 1 (steps 0-23h): analysis quality — what actually happened today
p1 = aggregate_and_save_day(ds, analysis_date, AROME_DAILY_DIR, "analysis")

# Day 2 (steps 24-47h): forecast quality — prediction for tomorrow
p2 = aggregate_and_save_day(ds, forecast_date, AROME_DAILY_DIR, "forecast")

# Day 3 (steps 48-51h, partial): discard — too few hours for reliable daily
# Not saved.

print(f"\nProcessed:")
print(f"  Analysis ({analysis_date}): {p1}")
print(f"  Forecast ({forecast_date}): {p2}")

In [ ]:
# Clean up raw GRIB to save disk
if DELETE_RAW and raw_path.exists():
    raw_path.unlink()
    print(f"Deleted raw GRIB: {raw_path}")

---
## Cache status and loader

In [ ]:
# Show what's in the cache
cached = sorted(AROME_DAILY_DIR.glob("arome_daily_*.nc"))
print(f"AROME daily cache: {AROME_DAILY_DIR}")
print(f"  Files: {len(cached)}")
if cached:
    dates = [f.stem.replace('arome_daily_', '') for f in cached]
    print(f"  Range: {dates[0]} → {dates[-1]}")
    total_mb = sum(f.stat().st_size for f in cached) / 1e6
    print(f"  Size:  {total_mb:.0f} MB total ({total_mb/len(cached):.1f} MB/day)")

In [ ]:
def load_arome_daily_cache(
    start_date: date,
    end_date: date,
    cache_dir: Path = AROME_DAILY_DIR,
) -> xr.Dataset:
    """Load cached AROME daily grids for a date range.
    
    Returns a gridded xr.Dataset covering all of France.
    Pass through extract_parcel_forcing() for parcel-level extraction.
    
    Parameters
    ----------
    start_date, end_date : date range
    cache_dir : path to arome_daily directory
    
    Returns
    -------
    xr.Dataset with dims (valid_time, latitude, longitude)
    """
    files = []
    missing = []
    current = start_date
    while current <= end_date:
        path = cache_dir / f"arome_daily_{current.isoformat()}.nc"
        if path.exists():
            files.append(path)
        else:
            missing.append(current)
        current += timedelta(days=1)
    
    if missing:
        logger.warning(
            "%d missing AROME days: %s%s",
            len(missing), missing[:5], "..." if len(missing) > 5 else ""
        )
    
    if not files:
        raise FileNotFoundError(
            f"No AROME daily files in {cache_dir} for {start_date} → {end_date}"
        )
    
    ds = xr.open_mfdataset(files, combine="by_coords")
    logger.info(
        "Loaded %d AROME daily grids (%s → %s), %d missing",
        len(files), start_date, end_date, len(missing)
    )
    return ds


# === Example: how the full pipeline uses this ===
# from irrigator.atmospheric.forcing import extract_parcel_forcing
#
# # Load gridded AROME for the ERA5-Land gap
# arome_gap_ds = load_arome_daily_cache(era5_last + timedelta(days=1), TODAY)
#
# # Extract at parcel (same downscaling as ERA5-Land)
# arome_gap_forcing = extract_parcel_forcing(arome_gap_ds, parcel, terrain)
#
# # Stitch: ERA5-Land owns the past, AROME fills only the gap
# full_forcing = era5_forcing.concat(arome_gap_forcing)
#
# # Any new parcel can call extract_parcel_forcing on the same cached grid
# # — no additional download needed.

---
## Automation

Daily cron at 06:00 local (AROME 00Z available ~04:00 UTC):

```bash
# crontab -e
0 6 * * * cd /path/to/IrriGator && python -m irrigator.arome_daily >> logs/arome.log 2>&1
```

Storage budget: ~50-100 MB/day × 365 days = ~20-35 GB/year.
For the growing season only (Apr-Oct, 210 days): ~10-20 GB/year.

### Backfill missing days

AROME is archived 14 days on Météo-France. If you missed a day,
you can backfill within that window:

In [ ]:
# Backfill: fetch and process multiple past days
# Useful if the cron failed or you're starting fresh

BACKFILL_START = date.today() - timedelta(days=13)  # max 14 days back
BACKFILL_END = date.today() - timedelta(days=1)     # up to yesterday

print(f"Backfill range: {BACKFILL_START} → {BACKFILL_END}")

# current = BACKFILL_START
# while current <= BACKFILL_END:
#     out = AROME_DAILY_DIR / f"arome_daily_{current.isoformat()}.nc"
#     if out.exists():
#         print(f"  {current}: already cached")
#     else:
#         try:
#             raw = fetch_arome_france(current)
#             ds_day = standardize_forecast_to_era5_format(open_forecast(raw), source="arome")
#             # ... de-accumulate + aggregate_and_save_day ...
#             print(f"  {current}: fetched and processed")
#         except Exception as e:
#             print(f"  {current}: FAILED — {e}")
#     current += timedelta(days=1)